# Spark RCA — Root Cause Classification & Ablation Study

**Multi-class classifier to predict Spark application failure root causes from execution telemetry.**

| Label | Root Cause | Failure Type |
|-------|---------------------|---------------|
| 0 | Baseline | No failure |
| 1 | OOM | Behavioral/Hard |
| 2 | Data Skew | Behavioral |
| 3 | Serialization | Hard |
| 4 | Network Timeout | Behavioral |
| 5 | Disk Space | Hard |
| 6 | Metadata | Hard |

**Pipeline:** Feature Extraction (25 features) → LR / DT / RF comparison → Ablation Study → Model C (22 features) export


---
## 1. Setup & Configuration


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, count, when, round as spark_round, isnan, isnull
from pyspark.sql.types import StringType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    RandomForestClassifier,
    LogisticRegression,
    DecisionTreeClassifier
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

print("✓ Imports loaded")


In [ ]:
# Configuration
FEATURES_PATH = "hdfs://namenode:8020/project/features"
MODEL_OUTPUT_PATH = "hdfs://namenode:8020/project/models/rca_rf_model"
RESULTS_OUTPUT_PATH = "hdfs://namenode:8020/project/predictions"

# 25 ML features extracted by FeatureExtractor.scala
FEATURE_COLUMNS = [
    # Stage-Level Aggregates (8)
    "mean_task_duration", "std_task_duration", "max_task_duration",
    "total_memory_spilled", "total_disk_spilled",
    "total_shuffle_read", "total_shuffle_write", "total_gc_time",
    # Structural Features (4)
    "total_stages", "failed_stages", "max_stage_parallelism", "stage_depth_of_failure",
    # Ratio Features (4)
    "completed_stage_ratio", "failed_stage_ratio", "spill_per_stage", "gc_per_stage",
    # Derived Features (9)
    "duration_heterogeneity_ratio", "duration_variance", "max_min_duration_ratio",
    "spill_ratio", "disk_spill_ratio", "peak_memory_ratio",
    "gc_time_ratio", "task_failure_rate", "retry_count"
]

LABEL_NAMES = {
    0: "Baseline", 1: "OOM", 2: "Data Skew",
    3: "Serialization Failure", 4: "Network Timeout",
    5: "Disk Space Failure", 6: "Metadata Failure"
}

print("✓ Configuration set")
print(f"  Features: {len(FEATURE_COLUMNS)} columns")
print(f"  Labels:   {len(LABEL_NAMES)} classes")


In [ ]:
import os, socket

def running_in_docker():
    if os.path.exists("/.dockerenv"):
        return True
    try:
        with open("/proc/1/cgroup", "r") as f:
            return "docker" in f.read() or "containerd" in f.read()
    except Exception:
        return False

if running_in_docker():
    HDFS_HOST = "namenode"
    MASTER = "local[*]"
    ENV = "Docker"
else:
    HDFS_HOST = "localhost"
    MASTER = "local[*]"
    ENV = "Host (Windows/Linux/macOS)"

spark = (
    SparkSession.builder
    .appName("RCA-ML-Classification")
    .master(MASTER)
    .config("spark.hadoop.fs.defaultFS", f"hdfs://{HDFS_HOST}:9000")
    .config("spark.sql.shuffle.partitions", "20")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"✓ Environment: {ENV}")
print(f"✓ Spark {spark.version} | Master: {spark.sparkContext.master}")


---
## 2. Data Loading & Train/Test Split


In [ ]:
# Load features from HDFS
df = spark.read.parquet(FEATURES_PATH)
df = df.fillna(0, subset=FEATURE_COLUMNS)

label_name_udf = udf(lambda l: LABEL_NAMES.get(l, "Unknown"), StringType())
df = df.withColumn("label_name", label_name_udf(col("label")))

total_rows = df.count()
print(f"✓ Loaded {total_rows} rows from HDFS")
print(f"  Features: {len(FEATURE_COLUMNS)}, Labels: {df.select('label').distinct().count()}")

# Label distribution
df.groupBy("label", "label_name").count().orderBy("label").show(truncate=False)

# 80/20 split
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.cache()
test_df = test_df.cache()
train_count = train_df.count()
test_count = test_df.count()

print(f"\nTrain: {train_count} ({100*train_count/total_rows:.0f}%)  |  Test: {test_count} ({100*test_count/total_rows:.0f}%)")


---
## 3. Model Training — Logistic Regression, Decision Tree, Random Forest

All models share: `VectorAssembler(25 features)` → `StandardScaler` → Classifier | 80/20 split, seed=42


In [ ]:
# Feature engineering pipeline (shared by all models)
assembler = VectorAssembler(inputCols=FEATURE_COLUMNS, outputCol="raw_features", handleInvalid="keep")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

# Shared evaluation function
def evaluate_model(predictions, model_name):
    """Evaluate with 4 standard multi-class metrics."""
    metrics = {}
    for metric_name, key in [("accuracy", "accuracy"), ("f1", "f1"),
                              ("weightedPrecision", "precision"), ("weightedRecall", "recall")]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label", predictionCol="prediction", metricName=metric_name)
        metrics[key] = evaluator.evaluate(predictions)

    print(f"\n📊 {model_name}:  Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"Prec={metrics['precision']:.4f}  Rec={metrics['recall']:.4f}")
    return metrics

print("✓ VectorAssembler + StandardScaler + evaluate_model() ready")


In [ ]:
# ── Logistic Regression (Multinomial) ────────────────────
print("Training Logistic Regression...")

lr = LogisticRegression(
    labelCol="label", featuresCol="features",
    maxIter=100, family="multinomial",
    elasticNetParam=0.0, regParam=0.01
)
lr_pipeline = Pipeline(stages=[assembler, scaler, lr])
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)
lr_metrics = evaluate_model(lr_predictions, "Logistic Regression")


In [ ]:
# ── Decision Tree ──────────────────────────────────────────
print("Training Decision Tree...")

dt = DecisionTreeClassifier(
    labelCol="label", featuresCol="features",
    maxDepth=5, maxBins=32, impurity="entropy", seed=42
)
dt_pipeline = Pipeline(stages=[assembler, scaler, dt])
dt_model = dt_pipeline.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_metrics = evaluate_model(dt_predictions, "Decision Tree")


In [ ]:
# ── Random Forest (100 trees) ─────────────────────────────
print("Training Random Forest...")

rf = RandomForestClassifier(
    labelCol="label", featuresCol="features",
    numTrees=100, maxDepth=10, maxBins=32, seed=42
)
rf_pipeline = Pipeline(stages=[assembler, scaler, rf])
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)
rf_metrics = evaluate_model(rf_predictions, "Random Forest")


---
## 4. Model Comparison


In [ ]:
# Model comparison table
results = {
    "Logistic Regression": lr_metrics,
    "Decision Tree": dt_metrics,
    "Random Forest": rf_metrics
}

print(f"\n{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("=" * 67)

best_model_name, best_f1 = None, -1.0
for name, m in results.items():
    if m['f1'] > best_f1:
        best_f1, best_model_name = m['f1'], name
    print(f"{name:<25} {m['accuracy']:>10.4f} {m['precision']:>10.4f} "
          f"{m['recall']:>10.4f} {m['f1']:>10.4f}")

print("=" * 67)
print(f"★ Best Model: {best_model_name} (Weighted F1 = {best_f1:.4f})")


In [ ]:
# ── Confusion Matrices (all 3 models) ─────────────────────
def compute_confusion_matrix(predictions, model_name):
    cm_df = predictions.groupBy("label").pivot("prediction").count().fillna(0).orderBy("label")
    cm_pandas = cm_df.toPandas().set_index('label')
    for lbl in range(7):
        col_name = str(float(lbl))
        if col_name not in cm_pandas.columns:
            cm_pandas[col_name] = 0
    cm_pandas = cm_pandas.reindex(sorted(cm_pandas.columns, key=float), axis=1)
    cm_array = cm_pandas.values.astype(float)

    total_per_class = predictions.groupBy("label").count().collect()
    correct_per_class = predictions.filter(col("prediction") == col("label")).groupBy("label").count().collect()
    correct_dict = {row['label']: row['count'] for row in correct_per_class}

    print(f"\n{'─'*50}")
    print(f"  {model_name} — Per-Class Accuracy")
    print(f"{'─'*50}")
    class_accs = {}
    for row in sorted(total_per_class, key=lambda r: r['label']):
        label = int(row['label'])
        total = row['count']
        correct = correct_dict.get(row['label'], 0)
        acc = correct / total if total > 0 else 0
        name = LABEL_NAMES.get(label, 'Unknown')
        status = "✅" if acc >= 0.5 else "❌"
        print(f"  {status} Label {label} ({name:<20}): {correct}/{total} = {acc:.0%}")
        class_accs[label] = acc
    return cm_array, class_accs

lr_cm, lr_class_acc = compute_confusion_matrix(lr_predictions, "Logistic Regression")
dt_cm, dt_class_acc = compute_confusion_matrix(dt_predictions, "Decision Tree")
rf_cm, rf_class_acc = compute_confusion_matrix(rf_predictions, "Random Forest")

# Plot side by side
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
short_names = [LABEL_NAMES[i][:8] for i in range(7)]

for ax, (cm, title) in zip(axes, [(lr_cm, "Logistic Regression"), (dt_cm, "Decision Tree"), (rf_cm, "Random Forest")]):
    if cm.shape[0] < 7 or cm.shape[1] < 7:
        padded = np.zeros((7, 7))
        padded[:cm.shape[0], :cm.shape[1]] = cm
        cm = padded
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    ax.set_xticks(range(7)); ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(7)); ax.set_yticklabels(short_names, fontsize=8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = int(cm[i, j])
            if val > 0:
                ax.text(j, i, str(val), ha='center', va='center',
                        color='white' if val > cm.max()/2 else 'black', fontsize=10)

plt.suptitle('Confusion Matrices — All 3 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 5. Feature Importance (Random Forest)


In [ ]:
# Extract feature importance from RF (25-feature model)
importances = rf_model.stages[-1].featureImportances.toArray()
feature_imp = sorted(zip(FEATURE_COLUMNS, importances), key=lambda x: x[1], reverse=True)

print("\n🌲 Random Forest — Feature Importance")
print(f"{'Rank':<6} {'Feature':<30} {'Importance':>12}")
print("-" * 50)
for rank, (feat, imp) in enumerate(feature_imp, 1):
    bar = '█' * int(imp * 50)
    star = " ⭐" if rank <= 5 else ""
    print(f"{rank:<6} {feat:<30} {imp:>10.4f}  {bar}{star}")

print(f"\nTop 5:  {sum(imp for _, imp in feature_imp[:5]):.1%}")
print(f"Top 10: {sum(imp for _, imp in feature_imp[:10]):.1%}")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 8))
top_n = 15
feat_names = [f[0] for f in feature_imp[:top_n]]
feat_vals = [f[1] for f in feature_imp[:top_n]]
colors = ['#1565C0' if i < 5 else '#90CAF9' for i in range(top_n)]
ax.barh(range(top_n), feat_vals, color=colors, edgecolor='white')
ax.set_yticks(range(top_n)); ax.set_yticklabels(feat_names, fontsize=10)
ax.invert_yaxis(); ax.set_xlabel('Importance')
ax.set_title('Random Forest — Top 15 Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 6. 5-Fold Cross-Validation (Random Forest)


In [ ]:
# 5-Fold Cross-Validation on the FULL dataset
print("Running 5-Fold Cross-Validation on Random Forest (25 features)...\n")

cv_assembler = VectorAssembler(inputCols=FEATURE_COLUMNS, outputCol="raw_features", handleInvalid="keep")
cv_scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
cv_rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=10, seed=42)
cv_pipeline = Pipeline(stages=[cv_assembler, cv_scaler, cv_rf])

paramGrid = ParamGridBuilder().build()

cv_results = {}
for metric_name in ["f1", "accuracy"]:
    evaluator_cv = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName=metric_name)
    crossval = CrossValidator(
        estimator=cv_pipeline, estimatorParamMaps=paramGrid,
        evaluator=evaluator_cv, numFolds=5, seed=42)
    cv_model = crossval.fit(df)
    cv_results[metric_name] = cv_model.avgMetrics[0]
    print(f"  5-Fold CV {metric_name.upper():<10}: {cv_results[metric_name]:.4f}")

print(f"\n  Test F1:  {rf_metrics['f1']:.4f}  vs  CV F1: {cv_results['f1']:.4f}")
print("  ✅ Not overfitting" if cv_results['f1'] >= rf_metrics['f1'] else "  ⚠️ Possible overfitting")


---
## 7. Ablation Study — Confound Analysis

Tests whether the classifier learns **runtime execution behavior** or merely memorises **query fingerprints**.

| Model | Features removed | Count |
|-------|-----------------|-------|
| **Model A** | None (baseline) | 25 |
| **Model B** | `total_stages`, `stage_depth_of_failure` | 23 |
| **Model C** | + `peak_memory_ratio` | 22 |

> **Hypothesis (H₀):** Removing confounded features does **not** degrade F1.


In [ ]:
# ============================================================
# ABLATION STUDY — 3 RF models with progressively fewer features
# ============================================================

CONFOUND_FEATURES = ["total_stages", "stage_depth_of_failure", "peak_memory_ratio"]

FEATURES_A = FEATURE_COLUMNS[:]                                     # 25
FEATURES_B = [f for f in FEATURE_COLUMNS                            # 23
              if f not in ["total_stages", "stage_depth_of_failure"]]
FEATURES_C = [f for f in FEATURE_COLUMNS                            # 22
              if f not in CONFOUND_FEATURES]

ablation_configs = [
    ("Model A (25 features)", FEATURES_A),
    ("Model B (23 features)", FEATURES_B),
    ("Model C (22 features)", FEATURES_C),
]

print(f"Feature sets: A={len(FEATURES_A)}, B={len(FEATURES_B)}, C={len(FEATURES_C)}")

# ── Helper: train + evaluate one RF ─────────────────────────
def run_ablation_model(name, feature_cols, train, test):
    asm = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="keep")
    scl = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
    rf  = RandomForestClassifier(labelCol="label", featuresCol="features",
                                  numTrees=100, maxDepth=10, seed=42)
    pipe = Pipeline(stages=[asm, scl, rf])
    model = pipe.fit(train)
    preds = model.transform(test)
    metrics = {}
    for mn, key in [("accuracy","accuracy"),("f1","f1"),
                     ("weightedPrecision","precision"),("weightedRecall","recall")]:
        ev = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=mn)
        metrics[key] = ev.evaluate(preds)
    return model, preds, metrics

# ── 5-fold CV helper ────────────────────────────────────────
def run_cv_f1(feature_cols, full_df):
    asm = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="keep")
    scl = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
    rf  = RandomForestClassifier(labelCol="label", featuresCol="features",
                                  numTrees=100, maxDepth=10, seed=42)
    pipe = Pipeline(stages=[asm, scl, rf])
    ev   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    cv   = CrossValidator(estimator=pipe, estimatorParamMaps=ParamGridBuilder().build(),
                          evaluator=ev, numFolds=5, seed=42)
    return cv.fit(full_df).avgMetrics[0]

# ── Train all 3 models ──────────────────────────────────────
print("\n" + "=" * 70)
print("ABLATION STUDY — CONFOUND ANALYSIS")
print("=" * 70)

abl_results, abl_models, abl_preds = {}, {}, {}

for name, feat_cols in ablation_configs:
    print(f"\n  Training {name}...")
    mdl, preds, mets = run_ablation_model(name, feat_cols, train_df, test_df)
    abl_results[name] = mets
    abl_models[name]  = mdl
    abl_preds[name]   = preds
    print(f"    Acc={mets['accuracy']:.4f}  F1={mets['f1']:.4f}  "
          f"Prec={mets['precision']:.4f}  Rec={mets['recall']:.4f}")

# ── 5-fold CV for all 3 ─────────────────────────────────────
print("\n" + "-" * 70)
print("5-Fold Cross-Validation (A / B / C)")
print("-" * 70)

abl_cv = {}
for name, feat_cols in ablation_configs:
    print(f"  CV for {name}...", end=" ")
    abl_cv[name] = run_cv_f1(feat_cols, df)
    print(f"CV F1 = {abl_cv[name]:.4f}")

# ── Comparison table ────────────────────────────────────────
print("\n" + "=" * 70)
print("ABLATION COMPARISON TABLE")
print("=" * 70)

base_key = ablation_configs[0][0]
base_f1  = abl_results[base_key]["f1"]

hdr = f"{'Model':<27} {'Acc':>7} {'F1':>7} {'Prec':>7} {'Rec':>7} {'CV F1':>7} {'ΔF1':>7}"
print("\n  " + hdr)
print("  " + "-" * len(hdr))
for name, _ in ablation_configs:
    m  = abl_results[name]
    cv = abl_cv[name]
    d  = m['f1'] - base_f1
    ds = f"{d:+.4f}" if name != base_key else "  base"
    print(f"  {name:<27} {m['accuracy']:>7.4f} {m['f1']:>7.4f} "
          f"{m['precision']:>7.4f} {m['recall']:>7.4f} {cv:>7.4f} {ds:>7}")

# ── Conclusion ──────────────────────────────────────────────
model_c_key = ablation_configs[2][0]
drop = base_f1 - abl_results[model_c_key]["f1"]

print(f"\n  F1 delta (A→C): {drop:+.4f}")
if drop <= 0.005:
    print("  ✅ H₀ SUPPORTED — behavioral features carry all discriminative signal.")
    print("  → Model C (22 features) is the recommended production model.")
else:
    print(f"  ⚠️  F1 drop ({drop:.4f}) > 0.005 — confounds may carry genuine signal.")

# Store Model C for export
model_c            = abl_models[model_c_key]
model_c_predictions = abl_preds[model_c_key]
model_c_metrics    = abl_results[model_c_key]
print(f"\n  Model C stored for export.")
print("=" * 70)


---
## 8. Model Export — Model C (22 Behavioral Features)

Model C is the ablation-validated production model. It excludes `total_stages`, `stage_depth_of_failure`,
and `peak_memory_ratio` — features that encode query identity rather than failure behavior.


In [ ]:
# Save Model C (22 features) to HDFS
model_c.write().overwrite().save(MODEL_OUTPUT_PATH)
print(f"✅ Model C (22 features) saved to HDFS: {MODEL_OUTPUT_PATH}")

# Save predictions
model_c_predictions.select(
    "app_id", "label", "prediction", "features", "probability"
).write.mode("overwrite").parquet(RESULTS_OUTPUT_PATH)
print(f"✅ Predictions saved to HDFS: {RESULTS_OUTPUT_PATH}")

print(f"\n  Exported model: 22 features")
print(f"  Accuracy: {model_c_metrics['accuracy']:.4f}")
print(f"  F1:       {model_c_metrics['f1']:.4f}")


In [ ]:
spark.stop()
print("✓ Spark session stopped")
